In [1]:
import joblib
import pandas as pd 
import numpy as np

In [2]:
# Load the model from the file
loaded_model = joblib.load('models/pacman_random_forest_v2.joblib')
scaler = joblib.load('scaler/pacman_random_forest_v2_scaler.joblib')

if hasattr(loaded_model, 'set_params') and hasattr(loaded_model, 'n_jobs'):
    loaded_model.set_params(n_jobs=1)

test = pd.read_csv('data/test_data.csv')
test.label.value_counts()

label
right    123
left     117
up        67
down      52
Name: count, dtype: int64

In [3]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)


def eval_model(df: pd.DataFrame, model=loaded_model, scaler=scaler):
    X = df.drop(columns=["label"]).copy()
    y_true = df["label"]

    if hasattr(model, "set_params") and hasattr(model, "n_jobs"):
        model.set_params(n_jobs=1)

    if scaler is not None and hasattr(scaler, "feature_names_in_"):
        scaler_features = list(scaler.feature_names_in_)
        missing_features = [name for name in scaler_features if name not in X.columns]
        if missing_features:
            raise ValueError(f"Missing features for scaler: {missing_features}")
        X = X[scaler_features]

    if scaler is not None:
        X_scaled = scaler.transform(X)
        X_eval = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
    else:
        X_eval = X

    if hasattr(model, "feature_names_in_"):
        model_features = list(model.feature_names_in_)
        missing_features = [name for name in model_features if name not in X_eval.columns]
        if missing_features:
            raise ValueError(f"Missing features for model: {missing_features}")
        X_eval = X_eval[model_features]

    y_pred = model.predict(X_eval)
    labels = sorted(set(y_true) | set(y_pred))

    eval_dict = {}
    eval_dict["accuracy"] = accuracy_score(y_true, y_pred)
    eval_dict["precision_macro"] = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    eval_dict["recall_macro"] = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    eval_dict["f1_macro"] = f1_score(y_true, y_pred, average="macro", zero_division=0)
    eval_dict["confusion"] = confusion_matrix(y_true, y_pred, labels=labels)
    eval_dict["labels"] = labels
    eval_dict["report"] = classification_report(y_true, y_pred, zero_division=0)
    eval_dict["predictions"] = y_pred

    return eval_dict

In [4]:
result = eval_model(test)
result

{'accuracy': 0.9972144846796658,
 'precision_macro': 0.9952830188679245,
 'recall_macro': 0.9979674796747967,
 'f1_macro': 0.9965986394557823,
 'confusion': array([[ 52,   0,   0,   0],
        [  0, 117,   0,   0],
        [  1,   0, 122,   0],
        [  0,   0,   0,  67]]),
 'labels': ['down', 'left', 'right', 'up'],
 'report': '              precision    recall  f1-score   support\n\n        down       0.98      1.00      0.99        52\n        left       1.00      1.00      1.00       117\n       right       1.00      0.99      1.00       123\n          up       1.00      1.00      1.00        67\n\n    accuracy                           1.00       359\n   macro avg       1.00      1.00      1.00       359\nweighted avg       1.00      1.00      1.00       359\n',
 'predictions': array(['left', 'left', 'left', 'left', 'left', 'left', 'left', 'left',
        'up', 'up', 'up', 'up', 'up', 'up', 'up', 'up', 'up', 'up', 'up',
        'up', 'up', 'up', 'up', 'up', 'up', 'up', 'up', 'l

In [5]:
pd.DataFrame(result['confusion'], index=result['labels'], columns=result['labels'])

,down,left,right,up
down,52,0,0,0
left,0,117,0,0
right,1,0,122,0
up,0,0,0,67


In [6]:
print(f"Accuracy on test data: {result['accuracy']}")
print("-"*20)
print(f"Precision on test data: {result['precision_macro']}")
print("-"*20)
print(f"Recall on test data: {result['recall_macro']}")
print("-"*20)
print(f"F1 on test data: {result['f1_macro']}")
print("-"*20)
print(f"Full Report on test data: \n{result['report']}")

Accuracy on test data: 0.9972144846796658
--------------------
Precision on test data: 0.9952830188679245
--------------------
Recall on test data: 0.9979674796747967
--------------------
F1 on test data: 0.9965986394557823
--------------------
Full Report on test data: 
              precision    recall  f1-score   support

        down       0.98      1.00      0.99        52
        left       1.00      1.00      1.00       117
       right       1.00      0.99      1.00       123
          up       1.00      1.00      1.00        67

    accuracy                           1.00       359
   macro avg       1.00      1.00      1.00       359
weighted avg       1.00      1.00      1.00       359

